In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Datos de entrenamiento
# Definimos frases positivas (1) y negativas (0)
frases = [
    "Me encanta este producto, es increíble", 
    "Excelente calidad y muy rápido",
    "Es lo mejor que he comprado nunca", 
    "Muy feliz con mi compra",
    "No me gusta nada, es un desastre", 
    "Pésimo servicio y llegó roto",
    "Una pérdida de dinero total", 
    "No lo recomiendo para nada, muy malo"
]

# 1 = Positivo, 0 = Negativo
etiquetas = [1, 1, 1, 1, 0, 0, 0, 0]

# Convertir las etiquetas a un array de Numpy (necesario para TensorFlow)
etiquetas = np.array(etiquetas)

# 2. Tokenización y Secuenciación
# Creamos el objeto Tokenizer para convertir palabras en números
tokenizer = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer.fit_on_texts(frases)
word_index = tokenizer.word_index

# Convertimos el texto a secuencias numéricas
secuencias = tokenizer.texts_to_sequences(frases)

# Aplicamos Padding (relleno) para que todas las frases tengan la misma longitud
secuencias_acolchadas = pad_sequences(secuencias, padding='post')

# 3. Mostrar resultados de la preparación
print("Índice de palabras:", word_index)
print("\nFrases convertidas a números con relleno (Padding):")
print(secuencias_acolchadas)
print("\nEtiquetas correspondientes:", etiquetas)

Índice de palabras: {'<OOV>': 1, 'es': 2, 'muy': 3, 'me': 4, 'y': 5, 'lo': 6, 'no': 7, 'nada': 8, 'encanta': 9, 'este': 10, 'producto': 11, 'increíble': 12, 'excelente': 13, 'calidad': 14, 'rápido': 15, 'mejor': 16, 'que': 17, 'he': 18, 'comprado': 19, 'nunca': 20, 'feliz': 21, 'con': 22, 'mi': 23, 'compra': 24, 'gusta': 25, 'un': 26, 'desastre': 27, 'pésimo': 28, 'servicio': 29, 'llegó': 30, 'roto': 31, 'una': 32, 'pérdida': 33, 'de': 34, 'dinero': 35, 'total': 36, 'recomiendo': 37, 'para': 38, 'malo': 39}

Frases convertidas a números con relleno (Padding):
[[ 4  9 10 11  2 12  0]
 [13 14  5  3 15  0  0]
 [ 2  6 16 17 18 19 20]
 [ 3 21 22 23 24  0  0]
 [ 7  4 25  8  2 26 27]
 [28 29  5 30 31  0  0]
 [32 33 34 35 36  0  0]
 [ 7  6 37 38  8  3 39]]

Etiquetas correspondientes: [1 1 1 1 0 0 0 0]


In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# --- Configuración de Hiperparámetros ---

# 1. Define cuántas palabras distintas va a recordar el modelo.
# "Solo quédate con las 500 palabras más frecuentes". Las muy raras las ignorará.
vocab_size = 500

# 2. Define la longitud máxima de las frases (vector de entrada).
max_length = 10

# 3. Indica que si una frase excede las 10 palabras se cortará el final ('post').
trunc_type = 'post'

# 4. Si una frase no llega a 10 palabras, se completará con 0 al final ('post').
padding_type = 'post'

# 5. Token OOV (Out of Vocabulary): para palabras no vistas en el entrenamiento.
oov_tok = "<OOV>"

# --- Ejecución del Proceso ---

# Inicializamos el Tokenizer con el tamaño de vocabulario y el token especial
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)

# Aprendizaje: Lee todas las frases y crea un diccionario interno de frecuencias.
tokenizer.fit_on_texts(frases)

# Transformación: Convierte las listas de palabras en listas de números.
# Ejemplo: "Me encanta el producto" -> [1, 2, 4, 3]
sequences = tokenizer.texts_to_sequences(frases)

# Homogeneización: Aplicamos el relleno y recorte para que todos tengan el mismo tamaño (10).
padded = pad_sequences(
    sequences, 
    maxlen=max_length, 
    padding=padding_type, 
    truncating=trunc_type
)

# --- Verificación ---
print("Forma de la matriz resultante (frases, palabras):", padded.shape)
print("\nEjemplo de frase procesada (con padding):")
print(padded[0])

Forma de la matriz resultante (frases, palabras): (8, 10)

Ejemplo de frase procesada (con padding):
[ 4  9 10 11  2 12  0  0  0  0]


In [4]:
import tensorflow as tf

# --- Definición de la Arquitectura del Modelo ---

model = tf.keras.Sequential([
    # 1. Capa de Embedding: Crea el mapa de significados vectoriales.
    # El modelo aprende a situar palabras con significados similares en posiciones cercanas.
    tf.keras.layers.Embedding(vocab_size, 16, input_length=max_length),
    
    # 2. Capa LSTM Bidireccional: "Entiende" el orden y el contexto.
    # Al ser bidireccional, procesa la frase de principio a fin y de fin a principio.
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    
    # 3. Capa Densa Intermedia: 16 neuronas para procesar las características extraídas.
    tf.keras.layers.Dense(16, activation='relu'),
    
    # 4. Capa de Salida: Una sola neurona con activación Sigmoid.
    # Devuelve un valor entre 0 y 1 (ideal para clasificación binaria: Positivo/Negativo).
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# --- Configuración del Entrenamiento (Compilación) ---

# loss='binary_crossentropy': Medida matemática del error para clasificación de dos clases.
# optimizer='adam': El algoritmo que ajusta los pesos de las neuronas para reducir el error.
# metrics=['accuracy']: Nos indica el porcentaje de aciertos durante el entrenamiento.

model.compile(
    loss='binary_crossentropy', 
    optimizer='adam', 
    metrics=['accuracy']
)

# Visualización de la estructura del modelo
print("--- Resumen de la Arquitectura del Modelo ---")
model.summary()

--- Resumen de la Arquitectura del Modelo ---


/home/ciabd12/anaconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
W0000 00:00:1775570642.383535  107943 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Entrenamos (con pocos datos necesitamos muchas épocas)
model.fit(padded, etiquetas, epochs=30, verbose=0)
print("Modelo entrenado.")

Modelo entrenado.
